# TabLint demo: Auto MPG
398 real cars (UCI Auto MPG, CC BY 4.0). Every number below comes from stored TabPFN-3.5 reports in `demo/video/`, so this runs without a GPU.

## 1. The real table, untouched
TabLint's top flag on the original data:

In [ ]:
import pandas as pd, json
import proofread
from proofread import Report

real = Report.load('../demo/video/auto_mpg.proofread.json')
real

In [ ]:
top = real.issues.iloc[0]
car = real.data.iloc[int(top.row)]
print(f"{car.car} ({car.model_year}): {int(car.cylinders)} cylinders, {car.displacement:.0f} cu in, {car.horsepower:.0f} hp, {car.mpg} mpg")
print(f"recorded weight {top.value:.0f} lb  ->  TabPFN-3.5 expects {top.suggested:.0f} lb (80%: {top.low:.0f}-{top.high:.0f})")
print('published curb weight: 4,762-4,775 lb (ultimatespecs.com); 4,900-5,000 lb (Wikipedia)')

## 2. Eight scripted typos
We planted eight realistic slips, listed in `answer_key.json`. A per-column z-score treats the Civic's pasted-in weight as normal:

In [ ]:
dirty = Report.load('../demo/video/auto_mpg_dirty.proofread.json')
w_col = dirty.data['weight']
z = (4354 - w_col.mean()) / w_col.std()
print(f'Honda Civic weight 4,354 lb: column z-score = {z:.1f}  (|z| > 3 would be needed to flag it)')
civ = dirty.issues[(dirty.issues.row == 181) & (dirty.issues.column == 'weight')].iloc[0]
print(f'TabLint: TabPFN-3.5 expects {civ.suggested:.0f} lb for a 4-cylinder, 53-hp, 33-mpg car (true value 1,795)')

In [ ]:
w = dirty.widget()      # click a red cell; accept / dismiss; keys n/p/a/d/u
w

## 3. How good are the suggested fixes?

In [ ]:
key = pd.DataFrame(json.load(open('../demo/video/answer_key.json')))
key[['car', 'column', 'recorded', 'true_value', 'suggested', 'proofread_rank', 'column_z_score', 'z_score_flags_it']]

In [ ]:
w.status = ['accepted' if (int(r['row']), r['column']) in set(zip(key.row, key.column)) else s for r, s in zip(w.issues, w.status)]
w.cleaned.loc[[181, 166, 175, 81], ['car', 'weight', 'displacement', 'horsepower', 'acceleration']]

## On your own data
```python
df = pd.read_csv('my_table.csv')
df.tablint.view(label='origin')
```